# Solving a model

Once a model is built, `model.solve(...)` hands it to a solver backend — HiGHS, Gurobi, CPLEX, CBC, GLPK, SCIP, Xpress, MOSEK, MindOpt, COPT, Knitro, or the GPU solver cuPDLPx — runs it, and writes the solution back into the variables. This page covers the everyday workflow:

- the one-step `model.solve(...)` call,
- inspecting the solver afterwards via `model.solver` and `SolverReport`,
- listing installed and licensed solvers.

For finer control — building a `Solver` yourself, adjusting the native solver model, or bracketing model transformations by hand — see [The Solver API](solver-api.ipynb).

## A small example model

We'll use a tiny LP throughout the page. Minimize $x + 2y$ subject to $x, y \ge 0$, $3x + 7y \ge 10$, $5x + 2y \ge 3$.

In [ ]:
import linopy
from linopy import Model


def build_model():
    m = Model()
    x = m.add_variables(lower=0, name="x")
    y = m.add_variables(lower=0, name="y")
    m.add_constraints(3 * x + 7 * y >= 10)
    m.add_constraints(5 * x + 2 * y >= 3)
    m.add_objective(x + 2 * y)
    return m

## One-step solving

`model.solve` picks the first available solver, runs it, writes the solution back into the variables, and returns a `(status, termination_condition)` tuple. You can specify which solver you want.

In [ ]:
m = build_model()
status, termination = m.solve(solver_name="highs", output_flag=False)
status, termination

In [ ]:
m.objective.value

In [ ]:
m.solution.to_pandas()

## After solving

After `model.solve(...)` the solver instance stays attached to the model as `model.solver`. You can read off the solver name, the native solver model, the status and — new in this release — a `SolverReport` with runtime, MIP gap, dual (best) bound, and iteration counts.

In [ ]:
m.solver

In [ ]:
m.solver.report

Not every backend fills in every field of `SolverReport` — if a solver doesn't expose a value it stays `None`. `mip_gap` and `dual_bound` are most informative on MIPs.

Some solvers (Gurobi, MOSEK, …) hold a license while the underlying handle is alive. You can release it explicitly:

In [ ]:
m.solver.close()  # frees the native handle (and license)
# or, to also detach the wrapper:
m.solver = None
m.solver, m.solver_name

## Available solvers

Two registries are exposed at the top level:

- `linopy.available_solvers` — solvers whose Python package or binary is **installed**. Cheap; does not acquire a license.
- `linopy.licensed_solvers` — the subset that currently passes a **license** probe. Useful in tests or to pick a solver at runtime.

In [ ]:
list(linopy.available_solvers)

In [ ]:
list(linopy.licensed_solvers)

Both are lazy and refreshable — call `linopy.available_solvers.refresh()` after installing or licensing a new solver in the same process. For a per-solver probe use `SolverClass.license_status()`, which returns a `LicenseStatus` dataclass:

In [ ]:
from linopy.solvers import Highs

Highs.license_status()